# Self Organising Maps

Using self-organising maps to visualise the feature space

In [ ]:
import sys

In [ ]:
%matplotlib inline
sys.path.append("..")
import wavelet_transform.utils.training_data as train
from wavelet_transform.utils.save_plots import save_figure

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Set sampling rate and noise std
sampling_rate = 50e6
std_noise = 0.01

# Set length of data to keep
sample_time = 20 / 1.5e6
data_len = int(sample_time * sampling_rate)

# Parameters for the wavelet transform
wavelet = "db30"
levels = 7  # max level for the transform
mode = "zero"
best_basis = None
min_level = 0

freq_threshold = 15e6 / (sampling_rate / 2)


### Training data

Run bubble dynamics over a range of frequencies and intensities to generate a larger array of training data.


In [ ]:
# Get data, select which features to analyse
node_info = [
    "centre frequency",
    # "level",
]
classifying_metrics = [
    # "energy",
    "average energy",
    # "log energy",
    "log average energy",
    "crest factor",
    "abs skewness",
    "kurtosis",
    "entropy",
    "spectral entropy",
    # "level entropy",
    "spectral kurtosis"
]

In [ ]:
if node_info:
    include_node_info = True
else:
    include_node_info = False

wpt_df = train.get_training_data(
    wavelet,
    sampling_rate,
    levels,
    mode,
    std_noise,
    best_basis,
    min_level=min_level,
    freq_threshold=freq_threshold,
    metrics=classifying_metrics,
    include_node_info=include_node_info
)

### Training

Train algorithm on the generated data

In [ ]:
n_data_points = wpt_df.shape[0]
print("data points", n_data_points)
n_neurons = 5 * n_data_points ** (1 / 2)
print("neurons", n_neurons)

### Feature selection

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn_som.som import SOM

aspect_ratio = 1  # width / height of som
epochs = 15  # scan through the data more times
max_iterations = int(1e4)  # maximum number of iterations across all epochs

In [ ]:
n_metrics = len(classifying_metrics) + len(node_info)
data_train = wpt_df[classifying_metrics + node_info]

# Standardise data
scaler = StandardScaler()
data_train_normalised = scaler.fit_transform(data_train)

# Self organising maps
som_cols = int(np.sqrt(n_neurons * aspect_ratio))  # make a map with specified aspect ratio
som_rows = int(n_neurons / som_cols)
som_size = (som_rows, som_cols)  # set size
som_height, som_width = som_size
n_clusters = som_width * som_height
som = SOM(som_height, som_width, n_metrics, max_iter=max_iterations)
labels = som.fit_predict(data_train_normalised, epochs=epochs)
print("iterations", som.n_iter_)

# Cluster centres
centers = som.cluster_centers_
# display(centers)
centers = som.cluster_centers_.reshape((n_clusters, n_metrics))
# print(centers)
centers = scaler.inverse_transform(centers)

# print("centers:", centers)

In [ ]:
# Plotting heatmaps for each feature
columns = 3
rows = n_metrics / columns
if rows.is_integer() is False:
    rows = int(rows) + 1  # round up if there aren't enough rows
rows = max(2, rows)
fig, axes = plt.subplots(int(rows), columns, figsize=(12 * aspect_ratio, 12))

for i, feature in enumerate(classifying_metrics):
    row, col = i // columns, i % columns
    feature_values = centers[:, i].reshape(som_size)
    # print(centers[:, i])
    ax:plt.Axes = axes[row, col]
    img = ax.imshow(feature_values, cmap='turbo', aspect='equal')
    ax.set_title(f'{feature.capitalize()}', fontsize=25)
    ax.set_xticks([])
    ax.set_yticks([])
    cbar = fig.colorbar(img, ax=ax, shrink=0.8)
    ticks = np.linspace(feature_values.min(), feature_values.max(), 5)
    rounded_ticks = np.round(ticks, 2)
    cbar.set_ticks(rounded_ticks)
    cbar.set_label("Amplitude")

plt.tight_layout()
plt.show()

save_figure(fig, f"SOMs_{wavelet}")
print(f"saved as 'SOMs_{wavelet}'")